# PydanticOutputParser → `with_structured_output()`

책에서는 `PydanticOutputParser`로 **① 형식 지침을 프롬프트에 주입 → ② LLM이 JSON 텍스트 생성 → ③ 파서가 문자열을 Pydantic 객체로 변환**하는 방식을 사용했습니다.

현재 LangChain(v1)의 권장 방식은 **모델의 네이티브 구조화 출력(structured output)** 을 쓰는 것입니다.

| 구분 | 책의 방식 | 현재 권장 방식 |
|---|---|---|
| 스키마 전달 | `get_format_instructions()` 문자열을 프롬프트에 삽입 | `llm.with_structured_output(Schema)` — 스키마가 API 파라미터로 전달됨 |
| 형식 보장 | 모델이 지침을 "잘 따르기를" 기대 | 공급자(OpenAI 등)가 JSON Schema 준수를 강제 (strict) |
| 모델 초기화 | `ChatOpenAI(model_name=...)` | `init_chat_model("openai:...")` (공급자 교체가 쉬움) |
| 에이전트 | - | `create_agent(..., response_format=Schema)` |
| 스트리밍 | "지원하지 않음"이라고 서술됨 | **지원됨** (dict/TypedDict 스키마는 부분 결과가 점진적으로 스트리밍) |

`PydanticOutputParser` 자체는 `langchain_core`에 여전히 존재하며, **네이티브 구조화 출력을 지원하지 않는 모델**에서는 여전히 유효한 대안입니다(마지막 절 참고).

In [ ]:
# 최초 1회 설치 (LangChain v1 기준)
# %pip install -qU langchain langchain-openai langchain-classic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 의 OPENAI_API_KEY, LANGSMITH_API_KEY 를 불러옵니다.

# LangSmith 추적: 별도 헬퍼 없이 환경변수만 설정하면 자동으로 활성화됩니다.
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "CH03-OutputParser")

In [ ]:
from langchain.chat_models import init_chat_model

# 공급자 중립적인 모델 초기화 ("공급자:모델명")
# 다른 모델로 바꾸려면 문자열만 교체하면 됩니다. 예) "anthropic:claude-sonnet-4-5", "ollama:llama3.1"
llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

다음은 이메일 본문 예시입니다.

In [ ]:
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

## 1. 출력 파서 없이 사용하는 경우

`langchain_teddynote.messages.stream_response` 같은 외부 헬퍼 없이, `chain.stream()`이 반환하는 메시지 청크의 `.text` 속성을 그대로 출력하면 됩니다.
(v1부터 `.text`는 메서드가 아니라 **속성**입니다.)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "다음의 이메일 내용중 중요한 내용을 추출해 주세요.\n\n{email_conversation}"
)

chain = prompt | llm

output = ""
for chunk in chain.stream({"email_conversation": email_conversation}):
    print(chunk.text, end="", flush=True)
    output += chunk.text

결과는 사람이 읽기엔 좋지만, 프로그램에서 `보낸 사람`, `미팅 날짜` 등을 꺼내 쓰기는 어렵습니다. 이제 구조화된 출력을 받아 보겠습니다.

## 2. 스키마 정의

`Field(description=...)`은 여전히 중요합니다. 이 설명이 JSON Schema에 포함되어 모델에게 전달됩니다.
클래스 docstring 역시 스키마 설명으로 전달되므로 작성해 두는 것이 좋습니다.

In [ ]:
from pydantic import BaseModel, Field


class EmailSummary(BaseModel):
    """이메일에서 추출한 핵심 정보"""

    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

## 3. `with_structured_output()` — 현재 권장 방식

- 프롬프트에 `{format}` 같은 **형식 지침을 넣을 필요가 없습니다.** 스키마는 API 요청의 `response_format`(JSON Schema)으로 전달됩니다.
- `langchain-openai`의 기본 방식은 `method="json_schema"`(OpenAI Structured Outputs)이며, 결과는 **검증된 Pydantic 객체**로 반환됩니다.

In [ ]:
structured_llm = llm.with_structured_output(EmailSummary)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. 모든 값은 한국어로 작성하세요."),
        ("human", "{question}\n\nEMAIL CONVERSATION:\n{email_conversation}"),
    ]
)

# 파서 없이: prompt → (구조화 출력 모델)
chain = prompt | structured_llm

response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)
response

In [ ]:
# 반환 타입은 EmailSummary 객체입니다.
print(type(response))
print(response.person, "/", response.date)

# dict 로 변환하려면 Pydantic v2 의 model_dump() 를 사용합니다. (.dict() 는 v1 방식)
response.model_dump()

### 원본 응답과 파싱 오류를 함께 받기: `include_raw=True`

디버깅할 때 유용합니다. `raw`(원본 AIMessage), `parsed`(Pydantic 객체), `parsing_error`(오류 또는 None)를 담은 dict를 반환하며, 파싱에 실패해도 예외를 던지지 않습니다.

In [ ]:
chain_with_raw = prompt | llm.with_structured_output(EmailSummary, include_raw=True)

result = chain_with_raw.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

print("parsing_error:", result["parsing_error"])
print("token usage  :", result["raw"].usage_metadata)
result["parsed"]

## 4. 구조화 출력 + 스트리밍

책에는 "`.with_structured_output()`은 `stream()`을 지원하지 않는다"고 되어 있지만, 현재는 사실과 다릅니다.

- 스키마를 **TypedDict / JSON Schema(dict)** 로 주면, 필드가 채워지는 과정이 **부분 dict**로 점진적으로 스트리밍됩니다.
- 스키마를 **Pydantic 클래스**로 주면, 검증이 끝난 완성 객체가 반환되므로 스트리밍 시 사실상 마지막에 결과가 나옵니다.

TypedDict에서는 `Annotated[타입, 기본값(...은 필수), "설명"]` 형태로 설명을 붙입니다.

In [ ]:
from typing_extensions import Annotated, TypedDict


class EmailSummaryDict(TypedDict):
    """이메일에서 추출한 핵심 정보"""

    person: Annotated[str, ..., "메일을 보낸 사람"]
    email: Annotated[str, ..., "메일을 보낸 사람의 이메일 주소"]
    subject: Annotated[str, ..., "메일 제목"]
    summary: Annotated[str, ..., "메일 본문을 요약한 텍스트"]
    date: Annotated[str, ..., "메일 본문에 언급된 미팅 날짜와 시간"]


stream_chain = prompt | llm.with_structured_output(EmailSummaryDict)

for partial in stream_chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
):
    print(partial)

## 5. (v1 신규) 에이전트에서 구조화 출력: `create_agent(response_format=...)`

LangChain v1의 핵심 API인 `create_agent`도 같은 스키마를 그대로 받습니다. 도구(tool)를 사용하는 에이전트가 최종 답을 구조화된 형태로 내야 할 때 이 방식을 씁니다.
결과는 `result["structured_response"]`에 담깁니다.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],  # 도구 없이도 사용 가능합니다.
    system_prompt="이메일에서 주요 정보를 추출하는 어시스턴트입니다. 모든 값은 한국어로 작성하세요.",
    response_format=EmailSummary,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": email_conversation}]}
)
result["structured_response"]

## 6. (참고) 네이티브 구조화 출력이 없는 모델: `PydanticOutputParser`

일부 로컬/소형 모델은 JSON Schema 기반 구조화 출력이나 도구 호출을 지원하지 않습니다. 이런 경우에만 책의 방식(형식 지침 + 파서)을 사용합니다.
import 경로는 `langchain_core.output_parsers`이며, v1에서도 그대로 유지됩니다.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=EmailSummary)

legacy_prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
).partial(format=parser.get_format_instructions())

legacy_chain = legacy_prompt | llm | parser

legacy_chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)